# 特征匹配实践课：暴力匹配 · FLANN · RANSAC · 全景拼接

**Feature Matching Lab — from two sets of keypoints to one set of reliable correspondences**

本课使用真实照片（见 `data/数据来源.txt`）：

| 数据 | 内容 | 真值 | 用于 |
|---|---|---|---|
| `graf_img1/2/3` | Oxford 涂鸦墙：视角逐渐增大 | `graf_H1to2`、`graf_H1to3` | 任务 1–3 |
| `boat_img1/3/5` | Oxford 港口：缩放 + 旋转逐渐增大 | `boat_H1to3`、`boat_H1to5` | 任务 3 |
| `pano_left / pano_right` | 圣彼得堡涅瓦河畔，手持相机转动拍摄的一对照片 | 无 | 任务 4 全景拼接 |

真值单应矩阵 $H$ 把图 1 的点 $(x, y, 1)^T$ 映射到图 N（左乘 $H$ 后除以第三个分量）。**一对匹配是否正确：映射误差 $\lVert H p_1 - p_2\rVert < 5$ 像素**——整节课都用它当裁判。

**本课任务：**

* **FLANN、全景拼接**：只需**调用现成函数**（`cv2.FlannBasedMatcher`、`cv2.findHomography`、`cv2.warpPerspective`）；
* **暴力匹配、RANSAC**：做两种实现再对比——
  * **(A) 调用现成函数**：`cv2.BFMatcher`、`cv2.findHomography(..., cv2.RANSAC)`；
  * **(B) 按课件步骤自己实现**：只用 NumPy，**不调用**对应的现成函数；
  * **(C) 对比**：结果是否一致、速度差多少，以及在真实数据上的效果。

**要求**：下课前完成全部任务并提交本 Notebook（提交前 *Restart & Run All*，保留输出）。每个"问题"请**结合你自己运行得到的具体数字**回答。标 ★ 的为选做。

| 任务 | (A) 调用 | (B) 自己实现 | (C) 对比与实验 |
|---|---|---|---|
| 1 暴力匹配 | `cv2.BFMatcher` | 欧氏 / 汉明距离矩阵、最近邻与次近邻、比值检验、交叉检验 | 与 OpenCV 逐个一致性与速度；距离阈值 vs 比值检验；交叉检验；ORB + 汉明距离 |
| 2 FLANN | `cv2.FlannBasedMatcher` | — | — |
| 3 RANSAC | `cv2.findHomography` | 4 点求单应（DLT）、自适应 RANSAC、内点精修 | 与 OpenCV 对比内点数与精度；直接最小二乘；迭代次数 |
| 4 全景拼接 | `findHomography`、`warpPerspective` | — | — |
| 5 总结 | | | |

## 0. 公共工具（已给出，直接运行）

约定：坐标 $(x, y)$，$x$ 向右、$y$ 向下；数组索引 `img[y, x]`。

In [ ]:
import os, sys, time
import numpy as np
import cv2
import matplotlib
import matplotlib.pyplot as plt
print('Python', sys.version.split()[0], '| OpenCV', cv2.__version__, '| NumPy', np.__version__)
assert hasattr(cv2, 'SIFT_create'), '需要 OpenCV >= 4.4（Python 3.7 可安装 opencv-python-headless 4.5/4.6）'

DATA = 'data'
for cand in ['data', os.path.join('..', 'data'), os.path.join('学生包', 'data')]:
    if os.path.isdir(cand):
        DATA = cand
        break
print('数据目录:', os.path.abspath(DATA))

def load_gray(name):
    img = cv2.imread(os.path.join(DATA, name + '.png'), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise IOError('读不到 ' + name)
    return img

def load_H(seq, j):
    return np.loadtxt(os.path.join(DATA, '%s_H1to%d.txt' % (seq, j)))

def apply_H(H, pts):
    # 把 N x 2 的点用 3x3 单应矩阵映射（含齐次坐标除法）
    pts = np.asarray(pts, dtype=np.float64).reshape(-1, 2)
    q = np.hstack([pts, np.ones((len(pts), 1))]) @ H.T
    return q[:, :2] / q[:, 2:3]

def corner_error(H_est, H_gt, shape):
    # 图 1 四个角分别用估计的 H 与真值 H 映射，取平均距离（像素）
    h, w = shape[:2]
    c = np.array([[0, 0], [w, 0], [w, h], [0, h]], float)
    return float(np.linalg.norm(apply_H(H_est, c) - apply_H(H_gt, c), axis=1).mean())

def is_correct(p1, p2, H, eps=5.0):
    # p1, p2: 一一对应的 N x 2 点；返回长度 N 的 bool 数组
    return np.linalg.norm(apply_H(H, p1) - np.asarray(p2, float), axis=1) < eps

def sift_features(img, n=2000):
    k, d = cv2.SIFT_create(nfeatures=n).detectAndCompute(img, None)
    return np.float64([kk.pt for kk in k]), d

def draw_matches(img1, img2, p1, p2, ok=None, n_show=150, seed=0, title=''):
    # 左右拼接画匹配线：绿 = 正确，红 = 错误（ok 为 None 时全画黄色）
    a = img1 if img1.ndim == 3 else cv2.cvtColor(img1, cv2.COLOR_GRAY2RGB)
    b = img2 if img2.ndim == 3 else cv2.cvtColor(img2, cv2.COLOR_GRAY2RGB)
    h = max(a.shape[0], b.shape[0])
    canvas = np.zeros((h, a.shape[1] + b.shape[1], 3), np.uint8)
    canvas[:a.shape[0], :a.shape[1]] = a; canvas[:b.shape[0], a.shape[1]:] = b
    idx = np.random.RandomState(seed).permutation(len(p1))[:n_show]
    plt.figure(figsize=(12, 12 * h / canvas.shape[1] + 0.6)); plt.imshow(canvas)
    for i in idx:
        col = 'gold' if ok is None else ('lime' if ok[i] else 'red')
        plt.plot([p1[i, 0], p2[i, 0] + a.shape[1]], [p1[i, 1], p2[i, 1]], '-', color=col, lw=0.7)
    plt.title(title); plt.axis('off'); plt.show()

G1, G2, G3 = load_gray('graf_img1'), load_gray('graf_img2'), load_gray('graf_img3')
fig, ax = plt.subplots(1, 6, figsize=(16, 2.6))
for a_, (im, t) in zip(ax, [(G1, 'graf 1'), (G2, 'graf 2'), (G3, 'graf 3'), (load_gray('boat_img1'), 'boat 1'),
                            (load_gray('boat_img3'), 'boat 3'), (load_gray('boat_img5'), 'boat 5')]):
    a_.imshow(im, cmap='gray'); a_.set_title(t); a_.axis('off')
plt.tight_layout(); plt.show()

## 任务 1　暴力匹配（调用 + 自己实现 + 对比）

**(B) 自己实现**（只用 NumPy，**不许**调用 `cv2.BFMatcher`、`cv2.norm`、`scipy.spatial` 等）
1. `l2_dist_matrix(A, B)`：$N\times d$ 与 $M\times d$ 描述子之间的欧氏距离矩阵 $N\times M$。提示：$\lVert a-b\rVert^2 = \lVert a\rVert^2 + \lVert b\rVert^2 - 2a\cdot b$，用矩阵乘法一次算完，**不要写两重循环**；开方前把负数截断为 0。
2. `hamming_dist_matrix(A, B)`：ORB 的描述子是 $N\times 32$ 的 `uint8`（256 位）。先用 `np.unpackbits(A, axis=1)` 展开成 0/1，再用矩阵乘法数"不同的位数"：$\text{ham}(a,b)=a\cdot(1-b)+(1-a)\cdot b$。
3. `knn2(D)`：对距离矩阵每一行，返回最近邻下标 `j1`、最近距离 `d1`、次近距离 `d2`（提示：`np.argpartition` 或 `np.argsort`）。
4. `ratio_test(d1, d2, r)`：返回布尔数组 $d_1 < r\cdot d_2$。
5. `cross_check(D, j1)`：返回布尔数组——$b_{j_1(i)}$ 的最近邻恰好是 $a_i$（互为最近邻）。

**(A) 调用**：`cv2.BFMatcher(cv2.NORM_L2).knnMatch(dA, dB, k=2)`；`cv2.BFMatcher(cv2.NORM_L2, crossCheck=True).match(dA, dB)`。

**(C) 对比**（graf 1→2，SIFT 各 2000 个点）
1. 你的最近邻下标、最近距离与 OpenCV `knnMatch` 是否**逐个一致**？交叉检验得到的匹配集合与 `crossCheck=True` 是否完全相同？两者耗时各多少？
2. 列一张表：仅最近邻；距离阈值 $d_1<$ 100 / 150 / 200 / 250；比值检验 $r$ = 0.6 / 0.7 / 0.8 / 0.9；交叉检验；比值 0.8 + 交叉检验——每行给出**保留对数、其中正确的数量、正确率、保住了多少比例的正确匹配**（分母：仅最近邻时正确的数量）；
3. 用 ORB（`cv2.ORB_create(2000)`）+ 你的汉明距离重复"仅最近邻"和"比值 0.8"两行，并验证最近距离与 `cv2.BFMatcher(cv2.NORM_HAMMING)` 一致。
4. 画出"仅最近邻"和"比值 0.8 + 交叉检验"的匹配图（`draw_matches`，绿 = 正确，红 = 错误）。

In [ ]:
def l2_dist_matrix(A, B):
    # TODO (B)
    raise NotImplementedError

def hamming_dist_matrix(A, B):
    # TODO (B)
    raise NotImplementedError

def knn2(D):
    # TODO (B)：返回 j1, d1, d2（长度均为 N）
    raise NotImplementedError

def ratio_test(d1, d2, r=0.8):
    # TODO (B)
    raise NotImplementedError

def cross_check(D, j1):
    # TODO (B)
    raise NotImplementedError

In [ ]:
# TODO (C) 1：与 cv2.BFMatcher 逐个对比、计时

In [ ]:
# TODO (C) 2、3、4：筛选策略对比表；ORB + 汉明距离；匹配图

**问题 1**：(a) 你的暴力匹配与 OpenCV 是否逐个一致？速度差多少倍？距离矩阵为什么能"一次矩阵乘法"算完？
(b) 正确率都在 80% 左右时，"距离阈值"和"比值检验"各保住了多少比例的正确匹配？为什么一刀切的距离阈值不如比值检验（结合"纹理丰富的点距离普遍偏大"解释）？
(c) $r$ 从 0.8 调到 0.6，保留对数和正确率怎样变化？交叉检验单独用效果如何，与比值检验叠加后呢？
(d) ORB 匹配时为什么要用汉明距离？ORB 与 SIFT 比值检验后的正确率各是多少？

> 你的回答：

## 任务 2　FLANN（调用）

`cv2.FlannBasedMatcher(dict(algorithm=1, trees=5), dict(checks=c))`：随机 kd 树，最多检查 `checks` 个叶子（"近似"的来源）。

1. 在 graf 1→2（各 2000 个点）上，取 `checks` = 1、4、16、64、256：记录 `knnMatch(k=2)` 的耗时、**最近邻与任务 1 精确结果一致的比例**、比值检验（0.8）后的对数和正确率；同时记录 `cv2.BFMatcher` 的耗时。
2. **规模实验**：把图 2 的描述子当作"库"，库中再混入来自其他图像（graf 3、boat 1/3/5、两张全景图）的 SIFT 描述子，使库大小 M = 2000、10000、30000、全部；用图 1 的 2000 个描述子做查询，比较 BF 与 FLANN（checks = 32）的耗时。

In [ ]:
# TODO：checks 实验；库规模实验

## 任务 3　RANSAC 估计单应矩阵（调用 + 自己实现 + 对比）

**(B) 自己实现**（只用 NumPy，**不许**调用 `cv2.findHomography`、`cv2.getPerspectiveTransform`）
1. `homography_dlt(p, q)`：由 $n\ge4$ 对点求 $H$，使 $q \sim H p$。每对点 $(x,y)\to(u,v)$ 给出两行方程
   $[-x, -y, -1, 0, 0, 0, ux, uy, u]$ 与 $[0, 0, 0, -x, -y, -1, vx, vy, v]$；堆成 $A$（$2n\times9$），用 `np.linalg.svd(A)` 取最小奇异值对应的右奇异向量作为 $h$，排成 $3\times3$ 并除以 $h_{33}$。
2. `ransac_homography(p, q, t=3.0, conf=0.99, max_iter=2000, seed=0)`：按课件框架
   ① 随机抽 4 对（`np.random.default_rng(seed).choice(n, 4, replace=False)`）→ ② `homography_dlt` → ③ 映射误差 $\lVert H p - q\rVert < t$ 的为内点 → ④ 内点多于当前最优则更新，
   并用**自适应**公式更新迭代次数 $N=\dfrac{\log(1-q)}{\log\big(1-(1-\varepsilon)^4\big)}$（$\varepsilon = 1-$ 内点比例）→ 迭代次数到 $N$ 或 `max_iter` 即停 →
   ⑤ **精修**：用最优模型的全部内点再调用一次 `homography_dlt`；再用精修后的 $H$ **重新统计内点**、再精修一次（4 个点拟合的模型带噪声，会漏掉一部分本该是内点的匹配）。返回 `H, inlier_mask, n_iter`。

**(A) 调用**：`cv2.findHomography(p, q, cv2.RANSAC, 3.0)`，返回 `(H, mask)`。

**(C) 对比**：在 graf 1→2、graf 1→3、boat 1→3、boat 1→5 上，先做 SIFT + 比值 0.8 + 交叉检验，再分别用
① 全部匹配直接 `homography_dlt`（最小二乘，不剔除外点）、② 你的 RANSAC、③ OpenCV RANSAC 估计 $H$。每种方法记录：内点数、内点中正确的比例、**四角误差**（`corner_error`，与真值比较）；你的 RANSAC 再记录迭代次数，并与公式 $N$ 在真实外点率下的值比较。
最后对 boat 1→5 画出你的 RANSAC 的内点（绿）与外点（红），以及图 1 四个角经三种 $H$ 映射到图 5 的四边形。

★ 选做：给 `homography_dlt` 加上 Hartley 归一化（把点平移到质心、缩放到平均距离 $\sqrt2$），比较精修后的四角误差。

In [ ]:
def homography_dlt(p, q):
    # TODO (B)
    raise NotImplementedError

def ransac_homography(p, q, t=3.0, conf=0.99, max_iter=2000, seed=0):
    # TODO (B)：返回 H, inlier_mask (bool, 长度 n), n_iter
    raise NotImplementedError

In [ ]:
# TODO (C)：四组图像的三种方法对比；boat 1->5 可视化

**问题 2**：(a) 在四组图像上，直接最小二乘的四角误差是多少？你的 RANSAC 与 OpenCV 呢？为什么含 10%–30% 错误匹配时最小二乘会被"带偏"那么多？
(b) 你的 RANSAC 实际迭代了多少次？与按真实外点率代入公式得到的 $N$ 相比如何？外点率 70% 时需要多少次？
(c) 如果去掉第 ⑤ 步里的"重新统计内点"，内点数会怎样变化（可以自己试一下）？换随机种子后，graf 1→2 与 graf 1→3 的结果各有多大波动？graf 1→3 有时内点更多、四角误差却更大，这说明"内点多"一定等于"模型准"吗？

> 你的回答：

## 任务 4　全景拼接（调用）

按课件流程，把 `pano_right.jpg` 拼到 `pano_left.jpg` 右侧：
① SIFT 检测描述（灰度图）；② FLANN（trees = 5，checks = 50）`knnMatch(dR, dL, k=2)` + 比值检验 0.75；
③ `cv2.findHomography(pR, pL, cv2.RANSAC, 4.0)` 求右图到左图的 $H$；
④ 用 `cv2.perspectiveTransform` 求右图四角在左图坐标系中的位置，确定画布（左图 ∪ 该四边形的外接矩形，必要时用平移矩阵 $T$ 保证坐标为正），`cv2.warpPerspective(R, T @ H, (W, H))` 变换右图，左图用 $T$ 平移到画布上；
⑤ 合成：右图直接覆盖在左图上（右图有效区域内取右图像素，其余取左图像素）。

要求输出：比值检验后的匹配数、RANSAC 内点数、$H$；拼接结果；对照实验：用**全部**比值检验后的匹配直接最小二乘求 $H$（`cv2.findHomography(pR, pL, 0)`）再拼接一次。

In [ ]:
# TODO：全景拼接

**问题 3**：(a) 比值检验后有多少对匹配，RANSAC 保留了多少内点？$H$ 的 $h_{13}$ 约等于多少，它说明了什么？第三行为什么不是 $(0, 0, 1)$？
(b) 不用 RANSAC 直接最小二乘，拼接结果如何？
(c) 拼接结果中有没有接缝、重影或错位？结合"单应只对平面场景或相机纯转动严格成立"解释原因（提示：注意近处的船、远处的建筑和水面倒影；两张照片的曝光也不完全相同）。

> 你的回答：

## 任务 5　总结

1. 用你自己的数字填写下表（graf 1→2）：

| 流水线阶段 | 匹配对数 | 正确率 |
|---|---|---|
| 最近邻 | | |
| + 比值检验 0.8 | | |
| + 交叉检验 | | |
| + RANSAC | | |

2. 暴力匹配和 RANSAC：你自己的实现与 OpenCV 的主要差别分别是什么？各用一个数字说明。
3. 课件思考题：如果把 SIFT 换成 ORB，暴力匹配器和 FLANN 分别应该怎么设置？

> 你的回答：